# Import

In [ ]:
!pip install QuantLib
!pip install vectorbt
!pip install plotly
!pip install pandasdmx --upgrade
!pip install sdmx1 pandas

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import QuantLib as ql
import yfinance as yf
import sdmx
from pandas_datareader import data as web
from datetime import datetime
import vectorbt as vbt
from datetime import timedelta

ModuleNotFoundError: No module named 'yfinance'

# Fetch Data

## API Request Def

In [3]:
def get_usd_rate(start_date, end_date):
  # Codes FRED pour les taux souverains US
  tickers = {
      '1Y': 'DGS1',
  }

  # Récupération des données
  df = web.DataReader(list(tickers.values()), 'fred', start_date, end_date)

  # Renommage des colonnes
  df.columns = tickers.keys()
  df = df.dropna()
  return df

In [4]:
def get_eur_rate(start_date, end_date):
  # Connexion à la BCE
  client = sdmx.Client("ECB")

  # Paramètres de la requête
  keys = {
      'FREQ': 'B',
      'REF_AREA': 'U2',
      'CURRENCY': 'EUR',
      'PROVIDER_FM': '4F',
      'INSTRUMENT_FM': 'G_N_A',
      'DATA_TYPE_FM': ['SR_1Y']
  }
  params = {'startPeriod': start_date,
            'endPeriod': end_date
            }
  response = client.data(
      resource_id='YC',
      key=keys,
      params= params
  )

  # Conversion en DataFrame
  df = sdmx.to_pandas(response, datetime={"dim": "TIME_PERIOD"})
  # Aplatir les colonnes MultiIndex
  df.columns = df.columns.get_level_values('DATA_TYPE_FM')

  # Renommer les colonnes pour plus de clarté
  df = df.rename(columns={
      'SR_1Y': 'EUR_1Y'
  })
  df = df.dropna()
  return df

In [5]:
def get_eurusd_price(start_date, end_date):
  # Download data
  eurusd = yf.download("EURUSD=X", start=start_date, end=end_date)
  eurusd = eurusd[['Close', 'High', 'Low']].rename(columns={"Close": "EURUSD"})
  return eurusd

## Do Fetching

In [6]:
start_date = "2018-01-01"
end_date = "2025-06-13"

In [7]:
# Fetch data
df1 = get_eur_rate(start_date, end_date)
print("EUR Rates Shape:", df1.shape)
print("EUR Rates Head:\n", df1.head())

df2 = get_usd_rate(start_date, end_date)
print("USD Rates Shape:", df2.shape)
print("USD Rates Head:\n", df2.head())

eurusd = get_eurusd_price(start_date, end_date)
print("EUR/USD Shape:", eurusd.shape)
print("EUR/USD Head:\n", eurusd.head())

# Combine data
Data = pd.DataFrame(index=eurusd.index)  # Set index directly
Data['EURUSD'] = eurusd['EURUSD']
Data['High'] = eurusd['High']
Data['Low'] = eurusd['Low']
Data['EUR_Rate_1Y'] = df1['EUR_1Y']/100
Data['USD_Rate_1Y'] = df2['1Y']/100

# Basic data cleaning
Data = Data.dropna()  # Remove rows with missing values
print("Final Data Shape:", Data.shape)
print("Final Data Head:\n", Data.head())

EUR Rates Shape: (1898, 1)
EUR Rates Head:
 DATA_TYPE_FM    EUR_1Y
TIME_PERIOD           
2018-01-02   -0.712228
2018-01-03   -0.696930
2018-01-04   -0.678375
2018-01-05   -0.675266
2018-01-08   -0.675209
USD Rates Shape: (1863, 1)
USD Rates Head:
               1Y
DATE            
2018-01-02  1.83
2018-01-03  1.81
2018-01-04  1.82
2018-01-05  1.80
2018-01-08  1.79


/tmp/ipython-input-5-2196443973.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  eurusd = yf.download("EURUSD=X", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

EUR/USD Shape: (1940, 3)
EUR/USD Head:
 Price         EURUSD      High       Low
Ticker      EURUSD=X  EURUSD=X  EURUSD=X
Date                                    
2018-01-01  1.200495  1.201504  1.199904
2018-01-02  1.201158  1.208094  1.200855
2018-01-03  1.206345  1.206709  1.200495
2018-01-04  1.201043  1.209190  1.200495
2018-01-05  1.206884  1.208459  1.202154
Final Data Shape: (1832, 5)
Final Data Head:
               EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y
Date                                                              
2018-01-02  1.201158  1.208094  1.200855    -0.007122       0.0183
2018-01-03  1.206345  1.206709  1.200495    -0.006969       0.0181
2018-01-04  1.201043  1.209190  1.200495    -0.006784       0.0182
2018-01-05  1.206884  1.208459  1.202154    -0.006753       0.0180
2018-01-08  1.203746  1.205400  1.195972    -0.006752       0.0179


In [8]:
# First Chart: EUR USD Rates
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=Data.index, y=Data['EUR_Rate_1Y'], mode='lines', name='EUR Rate 1Y'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['USD_Rate_1Y'], mode='lines', name='USD Rate 1Y'))
fig1.update_layout(
    title='1Y EUR and 1Y USD Rate ',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Maturity',
    template='plotly_white'
)
fig1.show()
# Third Chart: EUR/USD
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=Data.index, y=Data['EURUSD'], mode='lines', name='EUR/USD'))
fig3.update_layout(
    title='EUR/USD Exchange Rate',
    xaxis_title='Date',
    yaxis_title='Price',
    legend_title='Exchange Rate',
    template='plotly_white'
)
fig3.show()

# Pricing Options

## European Simple Option

In [32]:
def european_fx_option_pricing(option_type, S, K, r_usd, r_eur, T, sigma, today):
    """
    Price a European FX option (e.g., EUR/USD) using Black-Scholes-Merton model.

    Parameters:
    - option_type: 'call' or 'put' (call on EUR/USD = right to buy EUR, sell USD)
    - S: Spot exchange rate (e.g., EUR/USD spot rate in USD per EUR)
    - K: Strike exchange rate
    - r_usd: USD interest rate (domestic rate for EUR/USD)
    - r_eur: EUR interest rate (foreign rate for EUR/USD)
    - T: Time to maturity in years
    - sigma: Volatility of the exchange rate
    - today: QuantLib Date object for evaluation date

    Returns:
    - Tuple of (option_price, delta, gamma, theta, vega)
    """
    # Input validation
    if S <= 0 or K <= 0:
        raise ValueError("Spot price (S) and strike price (K) must be positive.")
    if sigma <= 0:
        raise ValueError("Volatility (sigma) must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity (T) must be positive.")

    # 1. Set the evaluation date
    ql.Settings.instance().evaluationDate = today

    # 2. Define the option type
    if option_type.lower() == 'call':
        option_type_ql = ql.Option.Call  # Call on EUR/USD: right to buy EUR
    elif option_type.lower() == 'put':
        option_type_ql = ql.Option.Put  # Put on EUR/USD: right to sell EUR
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    # 3. Create the option object
    payoff = ql.PlainVanillaPayoff(option_type_ql, K)
    # Calculate maturity date (use calendar for accuracy)
    exercise_date = today + ql.Period(int(T * 360), ql.Days)  # Approximate T in days
    exercise = ql.EuropeanExercise(exercise_date)
    european_option = ql.VanillaOption(payoff, exercise)

    # 4. Define the market data
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    # Domestic rate (USD for EUR/USD)
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_usd, ql.Actual365Fixed()))
    # Foreign rate (EUR for EUR/USD)
    dividend_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_eur, ql.Actual365Fixed()))

    vol_handle = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), sigma, ql.Actual365Fixed()))

    # 5. Create the Black-Scholes process for FX
    bsm_process = ql.BlackScholesMertonProcess(spot_handle, dividend_handle, rate_handle, vol_handle)

    # 6. Create the pricing engine
    engine = ql.AnalyticEuropeanEngine(bsm_process)

    # 7. Set the pricing engine for the option
    european_option.setPricingEngine(engine)

    # 8. Calculate the option price and Greeks
    option_price = european_option.NPV()  # Price in USD per EUR
    delta = european_option.delta()  # Delta w.r.t. spot exchange rate
    gamma = european_option.gamma()  # Gamma w.r.t. spot exchange rate
    theta = european_option.theta() / 365  # Annualized theta (USD per EUR per year)
    vega = european_option.vega() / 100   # Vega per 1% change in volatility

    return option_price, delta, gamma, theta, vega


In [33]:
T = 1/12

In [34]:
def pricing_straddle(Data):
  # Initialize DataFrame to store straddle price and Greeks
  Option_Price_List = pd.DataFrame(
      columns=[f"Straddle_{int(T*12)}M_ATMF_USD", "Delta", "Gamma", "Theta", "Vega"],
      index=Data.index
  )

  vol = Data["EURUSD"].pct_change().dropna().std() * np.sqrt(252)  # Annualized volatility
  print(f"Calculated volatility: {vol}")

  # Iterate over DataFrame
  for index, row in Data.iterrows():
      S = row["EURUSD"]
      K = row["EURUSD"]*np.exp((row["USD_Rate_1Y"]-row["EUR_Rate_1Y"]+1/2*vol**2)*T)
      r = row["USD_Rate_1Y"]
      q = row["EUR_Rate_1Y"]
      sigma = vol
      today = ql.Date(index.day, index.month, index.year)

      try:
          # Price call and put options
          option_price_call_atm, delta_call, gamma_call, theta_call, vega_call = european_fx_option_pricing(
              "call", S, K, r, q, T, sigma, today
          )
          option_price_put_atm, delta_put, gamma_put, theta_put, vega_put = european_fx_option_pricing(
              "put", S, K, r, q, T, sigma, today
          )
          # Calculate straddle price and Greeks
          straddle_price = option_price_call_atm + option_price_put_atm
          straddle_delta = delta_call + delta_put
          straddle_gamma = gamma_call + gamma_put
          straddle_theta = theta_call + theta_put
          straddle_vega = vega_call + vega_put

          # Store in DataFrame
          Option_Price_List.loc[index] = [
              straddle_price,
              straddle_delta,
              straddle_gamma,
              straddle_theta,
              straddle_vega
          ]
      except Exception as e:
          print(f"Error pricing for date {index}: {e}")
          Option_Price_List.loc[index] = [None, None, None, None, None]

  for i in Option_Price_List.columns:
      Data[i] = Option_Price_List[i]
  return Data

Data = pricing_straddle(Data).copy()
print(Data.head())
print(Data.columns)

# Créer le graphique avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=Data.index, y=Data[f"Straddle_{int(T*12)}M_ATMF_USD"], mode='lines', name='Upper Bound'))

fig.update_layout(
    title="Option Price Over Time",
    xaxis_title="Date",
    yaxis_title="Option Price"
)

fig.show()

Calculated volatility: 0.07544634080618155
              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2018-01-02  1.201158  1.208094  1.200855    -0.007122       0.0183   
2018-01-03  1.206345  1.206709  1.200495    -0.006969       0.0181   
2018-01-04  1.201043  1.209190  1.200495    -0.006784       0.0182   
2018-01-05  1.206884  1.208459  1.202154    -0.006753       0.0180   
2018-01-08  1.203746  1.205400  1.195972    -0.006752       0.0179   

           Straddle_3M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2018-01-02             0.035988 -0.001191  30.728497 -0.000345  0.002749   
2018-01-03             0.036142 -0.001176  30.595975 -0.000346  0.002761   
2018-01-04             0.035982 -0.001173  30.730596 -0.000345  0.002749   
2018-01-05             0.036157 -0.001163  30.581778 -0.000346  0.002762   
2018-01-08

## Def Pivot Points

In [12]:
def Pivot_Points(Data, pivot_left, pivot_right):
        try:
            from collections import deque
            def clean_deque(i, k, deq, df, key, isHigh):
                if deq and deq[0] == i - k:
                    deq.popleft()
                if isHigh:
                    while deq and df.iloc[i][key] > df.iloc[deq[-1]][key]:
                        deq.pop()
                else:
                    while deq and df.iloc[i][key] < df.iloc[deq[-1]][key]:
                        deq.pop()

            data = Data[["High", "Low"]].copy()
            data['H'] = False
            data['L'] = False

            win_size = pivot_left + pivot_right + 1
            deqHigh = deque()
            deqLow = deque()

            max_idx = 0
            min_idx = 0
            i = 0
            j = pivot_left
            pivot_low = None
            pivot_high = None

            for index, row in data.iterrows():
                if i < win_size:
                    clean_deque(i, win_size, deqHigh, data, 'High', True)
                    clean_deque(i, win_size, deqLow, data, 'Low', False)
                    deqHigh.append(i)
                    deqLow.append(i)

                    if data.iloc[i]['High'] > data.iloc[max_idx]['High']:
                        max_idx = i
                    if data.iloc[i]['Low'] < data.iloc[min_idx]['Low']:
                        min_idx = i

                    if i == win_size - 1:
                        if data.iloc[max_idx]['High'] == data.iloc[j]['High']:
                            data.at[data.index[j], 'H'] = True
                            pivot_high = data.iloc[j]['High']
                        if data.iloc[min_idx]['Low'] == data.iloc[j]['Low']:
                            data.at[data.index[j], 'L'] = True
                            pivot_low = data.iloc[j]['Low']
                else:
                    j += 1
                    clean_deque(i, win_size, deqHigh, data, 'High', True)
                    clean_deque(i, win_size, deqLow, data, 'Low', False)
                    deqHigh.append(i)
                    deqLow.append(i)

                    if data.iloc[deqHigh[0]]['High'] == data.iloc[j]['High']:
                        data.at[data.index[j], 'H'] = True
                        pivot_high = data.iloc[j]['High']
                    if data.iloc[deqLow[0]]['Low'] == data.iloc[j]['Low']:
                        data.at[data.index[j], 'L'] = True
                        pivot_low = data.iloc[j]['Low']

                data.at[data.index[j], 'Last_High_Value'] = pivot_high
                data.at[data.index[j], 'Last_Low_Value'] = pivot_low
                i += 1

            # Low pivot calculation
            lows_list = []
            broken_lows = []
            first_value_low = True
            data["Low_Pivot"] = np.nan

            for idx, row in data.iterrows():
                lows_list = [x for x in lows_list if not np.isnan(x)]
                last_low = row['Last_Low_Value']
                low = row['Low']

                if pd.notna(last_low):
                    if first_value_low:
                        lows_list.append(last_low)
                        data.at[idx, "Low_Pivot"] = last_low
                        first_value_low = False
                        continue

                    if not lows_list:
                        if last_low not in broken_lows:
                            lows_list.append(last_low)
                            data.at[idx, "Low_Pivot"] = last_low
                    elif len(lows_list) > 1:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                        if last_low not in broken_lows:
                            if last_low != lows_list[-1]:
                                lows_list.append(last_low)
                            data.at[idx, "Low_Pivot"] = lows_list[-1]
                        else:
                            data.at[idx, "Low_Pivot"] = lows_list[-1]
                    else:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                            if last_low not in broken_lows:
                                lows_list.append(last_low)
                                data.at[idx, "Low_Pivot"] = last_low
                            else:
                                data.at[idx, "Low_Pivot"] = None
                        else:
                            if last_low not in broken_lows:
                                if last_low != lows_list[-1]:
                                    lows_list.append(last_low)
                                data.at[idx, "Low_Pivot"] = lows_list[-1]
                            else:
                                data.at[idx, "Low_Pivot"] = lows_list[-1]
                elif not first_value_low:
                    if lows_list:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                        data.at[idx, "Low_Pivot"] = lows_list[-1] if lows_list else None

                    # -------------------------------
                    # High pivot calculation
                    highs_list = []
                    broken_highs = []
                    first_value_high = True
                    data["High_Pivot"] = np.nan

                    for idx, row in data.iterrows():
                        highs_list = [x for x in highs_list if not np.isnan(x)]
                        last_high = row['Last_High_Value']
                        high = row['High']

                        if pd.notna(last_high):
                            if first_value_high:
                                highs_list.append(last_high)
                                data.at[idx, "High_Pivot"] = last_high
                                first_value_high = False
                                continue

                            if not highs_list:
                                if last_high not in broken_highs:
                                    highs_list.append(last_high)
                                    data.at[idx, "High_Pivot"] = last_high
                            elif len(highs_list) > 1:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                if last_high not in broken_highs:
                                    if last_high != highs_list[-1]:
                                        highs_list.append(last_high)
                                    data.at[idx, "High_Pivot"] = highs_list[-1]
                                else:
                                    data.at[idx, "High_Pivot"] = highs_list[-1]
                            else:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                    if last_high not in broken_highs:
                                        highs_list.append(last_high)
                                        data.at[idx, "High_Pivot"] = last_high
                                    else:
                                        data.at[idx, "High_Pivot"] = None
                                else:
                                    if last_high not in broken_highs:
                                        if last_high != highs_list[-1]:
                                            highs_list.append(last_high)
                                        data.at[idx, "High_Pivot"] = highs_list[-1]
                                    else:
                                        data.at[idx, "High_Pivot"] = highs_list[-1]
                        elif not first_value_high:
                            if highs_list:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                data.at[idx, "High_Pivot"] = highs_list[-1] if highs_list else None

            colonnes_a_supprimer = ['Last_High_Value', 'Last_Low_Value']
            data = data.drop(colonnes_a_supprimer, axis=1)

            Data["Low_Pivot"] = data["Low_Pivot"]
            Data["High_Pivot"] = data["High_Pivot"]

            return Data

        except Exception as e:
            raise

## Get Pivot Point Data

In [35]:
left = 10
right = 10
Data = Pivot_Points(Data, right, left)
print(Data.tail())
print(Data.columns)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=Data.index, y=Data['High'], mode='lines', name='EUR Rate 3M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['Low'], mode='lines', name='EUR Rate 3M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['High_Pivot'], mode='lines', name='EUR Rate 6M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['Low_Pivot'], mode='lines', name='EUR Rate 1Y'))
fig1.update_layout(
    title=f'Pivot Point EUR ({left},{right})',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Maturity',
    template='plotly_white'
)
fig1.show()

              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2025-06-06  1.145383  1.145869  1.137230     0.018325       0.0414   
2025-06-09  1.140784  1.144034  1.138758     0.018281       0.0413   
2025-06-10  1.142805  1.144833  1.137462     0.018161       0.0412   
2025-06-11  1.143746  1.149095  1.140602     0.017834       0.0408   
2025-06-12  1.150933  1.162872  1.150285     0.017840       0.0406   

           Straddle_3M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2025-06-06             0.034102  -0.00109  32.157498 -0.000327  0.002616   
2025-06-09             0.033966 -0.001088  32.287265 -0.000325  0.002606   
2025-06-10             0.034027 -0.001088  32.230493 -0.000326   0.00261   
2025-06-11             0.034058 -0.001085  32.204835 -0.000326  0.002612   
2025-06-12             0.034272 -0.001077  32.003707 

# Options Strategy Signal

In [14]:
def signals(Data):

    high_pivot_changed = Data["High_Pivot"] > Data["High_Pivot"].shift(1)
    low_pivot_changed = Data["Low_Pivot"] < Data["Low_Pivot"].shift(1)

    signals = []
    for i in range(len(Data)):
        if high_pivot_changed[i]:
            signals.append(1)
        elif low_pivot_changed[i]:
            signals.append(-1)
        else:
            signals.append(0)

    Data["Signals"] = signals
    return Data

Data = signals(Data)
value_counts = Data["Signals"].value_counts(dropna=False)
print(value_counts)

print(Data.tail())

Signals
 0    1739
 1      52
-1      41
Name: count, dtype: int64
              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2025-06-06  1.145383  1.145869  1.137230     0.018325       0.0414   
2025-06-09  1.140784  1.144034  1.138758     0.018281       0.0413   
2025-06-10  1.142805  1.144833  1.137462     0.018161       0.0412   
2025-06-11  1.143746  1.149095  1.140602     0.017834       0.0408   
2025-06-12  1.150933  1.162872  1.150285     0.017840       0.0406   

           Straddle_3M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2025-06-06             0.034102 -0.001882  18.510262 -0.000185  0.004518   
2025-06-09             0.033966 -0.001878  18.585092 -0.000185  0.004499   
2025-06-10             0.034027  -0.00188  18.552781 -0.000185  0.004508   
2025-06-11             0.034058 -0.001874  18.539006 -0.000185

/tmp/ipython-input-14-3888941173.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-14-3888941173.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



# Strategy

## Variables

In [36]:
start_date = "2018-01-01"
end_date = "2025-06-13"
Data = pd.DataFrame()
#--------------------------------------------------
df1 = get_eur_rate(start_date, end_date)
df2 = get_usd_rate(start_date, end_date)
eurusd = get_eurusd_price(start_date, end_date)
Data = pd.DataFrame(index=eurusd.index)
Data['EURUSD'] = eurusd['EURUSD']
Data['High'] = eurusd['High']
Data['Low'] = eurusd['Low']
Data['EUR_Rate_1Y'] = df1['EUR_1Y']/100
Data['USD_Rate_1Y'] = df2['1Y']/100
Data = Data.dropna()
#--------------------------------------------------
T = 1/12
Data = pricing_straddle(Data).copy()
#--------------------------------------------------
pivot_left = 10
pivot_right = 10
Data = Pivot_Points(Data, right, left)
#--------------------------------------------------
Data = signals(Data).copy()
#--------------------------------------------------
print("Final Data Shape:", Data.shape)
print("Final Data Head:\n", Data.tail())

/tmp/ipython-input-5-2196443973.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


Calculated volatility: 0.07544634080618155
Final Data Shape: (1832, 13)
Final Data Head:
               EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2025-06-06  1.145383  1.145869  1.137230     0.018325       0.0414   
2025-06-09  1.140784  1.144034  1.138758     0.018281       0.0413   
2025-06-10  1.142805  1.144833  1.137462     0.018161       0.0412   
2025-06-11  1.143746  1.149095  1.140602     0.017834       0.0408   
2025-06-12  1.150933  1.162872  1.150285     0.017840       0.0406   

           Straddle_1M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2025-06-06             0.019741  -0.00109  32.157498 -0.000327  0.002616   
2025-06-09             0.019662 -0.001088  32.287265 -0.000325  0.002606   
2025-06-10             0.019697 -0.001088  32.230493 -0.000326   0.00261   
2025-06-11             0.019714 -0.0010

/tmp/ipython-input-14-3888941173.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-14-3888941173.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



## Def Get Orders CFD/Futures (Entry, TP, SL)

In [87]:
def Get_Orders_Futures(index, price, signals, SL_Pct, TP_Pct, SIZE):
    order_id = 1
    orders = []
    i = 0

    # Ensure all inputs are Pandas Series
    index = pd.Series(index)
    price = pd.Series(price)
    signals = pd.Series(signals)

    while i < len(index):
        #---------------------------------------------------------------------------
        if signals.iloc[i] == 1:  # LONG SIGNAL
            entry_price = price.iloc[i]
            sl_price = entry_price * (1 - SL_Pct)
            tp_price = entry_price * (1 + TP_Pct)

            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': SIZE,
                'price': entry_price,
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': entry_price}
            })
            order_id += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            for j in range(i + 1, len(index)):
                future_idx = index.iloc[j]
                future_price = price.iloc[j]
                # First level hit triggers exit
                if future_price <= sl_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': -SIZE,
                        'price': sl_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif future_price >= tp_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': -SIZE,
                        'price': tp_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': -SIZE,
                    'price': price.iloc[-1],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': entry_price}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        #---------------------------------------------------------------------------
        elif signals.iloc[i] == -1:  # SHORT SIGNAL
            entry_price = price.iloc[i]
            sl_price = entry_price * (1 + SL_Pct)
            tp_price = entry_price * (1 - TP_Pct)

            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': -SIZE,
                'price': entry_price,
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': entry_price}
            })
            order_id += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            for j in range(i + 1, len(index)):
                future_idx = index.iloc[j]
                future_price = price.iloc[j]

                if future_price >= sl_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': SIZE,
                        'price': sl_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif future_price <= tp_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': SIZE,
                        'price': tp_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': SIZE,
                    'price': price.iloc[-1],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': entry_price}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        i += 1
        #---------------------------------------------------------------------------
    # Convert to DataFrame
    orders_df = pd.DataFrame(orders)
    orders_df['timestamp'] = pd.to_datetime(orders_df['timestamp'])
    orders_df = orders_df.groupby('timestamp').agg({'size': 'sum', 'price': 'first'}).sort_index()
    # You need to define "close" or replace it with the relevant series for reindexing
    # Here, I'll assume you want to use the same index for reindexing as provided at input
    sizes = orders_df['size'].reindex(index, fill_value=0)
    prices = orders_df['price'].reindex(index)

    return orders_df, sizes, prices

## Def get Orders Straddle ATMF Entry/SL/TP

In [ ]:
def Get_Orders_Straddle_ATMF(index, price, signals, SL_Pct, TP_Pct, SIZE):
    order_id = 1
    orders = []
    i = 0

    # Ensure all inputs are Pandas Series
    index = pd.Series(index)
    price = pd.Series(price)
    signals = pd.Series(signals)

    while i < len(index):
        #---------------------------------------------------------------------------
        if signals.iloc[i] == 1:  # LONG SIGNAL
            entry_price = price.iloc[i]
            sl_price = entry_price * (1 - SL_Pct)
            tp_price = entry_price * (1 + TP_Pct)

            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': SIZE,
                'price': entry_price,
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': entry_price}
            })
            order_id += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            for j in range(i + 1, len(index)):
                future_idx = index.iloc[j]
                future_price = price.iloc[j]
                # First level hit triggers exit
                if future_price <= sl_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': -SIZE,
                        'price': sl_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif future_price >= tp_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': -SIZE,
                        'price': tp_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': -SIZE,
                    'price': price.iloc[-1],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': entry_price}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        #---------------------------------------------------------------------------
        elif signals.iloc[i] == -1:  # SHORT SIGNAL
            entry_price = price.iloc[i]
            sl_price = entry_price * (1 + SL_Pct)
            tp_price = entry_price * (1 - TP_Pct)

            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': -SIZE,
                'price': entry_price,
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': entry_price}
            })
            order_id += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            for j in range(i + 1, len(index)):
                future_idx = index.iloc[j]
                future_price = price.iloc[j]

                if future_price >= sl_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': SIZE,
                        'price': sl_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif future_price <= tp_price:
                    orders.append({
                        'timestamp': future_idx,
                        'size': SIZE,
                        'price': tp_price,
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': entry_price}
                    })
                    order_id += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': SIZE,
                    'price': price.iloc[-1],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': entry_price}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        i += 1
        #---------------------------------------------------------------------------
    # Convert to DataFrame
    orders_df = pd.DataFrame(orders)
    orders_df['timestamp'] = pd.to_datetime(orders_df['timestamp'])
    orders_df = orders_df.groupby('timestamp').agg({'size': 'sum', 'price': 'first'}).sort_index()
    # You need to define "close" or replace it with the relevant series for reindexing
    # Here, I'll assume you want to use the same index for reindexing as provided at input
    sizes = orders_df['size'].reindex(index, fill_value=0)
    prices = orders_df['price'].reindex(index)

    return orders_df, sizes, prices

## Def Show Backesting Results

In [76]:
#-------------------------------------------------------------------------------
def show_backtesting_result(portfolio):
  # Create df_stats_reset Dataframe
  general_stats = portfolio.stats()
  assets = list(portfolio.wrapper.columns)
  asset1_stats = portfolio[assets[0]].stats()
  asset2_stats = portfolio[assets[1]].stats()
  all_stat_names = sorted(set(general_stats.index) | set(asset1_stats.index) | set(asset2_stats.index))
  data = {
      'General': [general_stats.get(stat, "") for stat in all_stat_names],
      assets[0]: [asset1_stats.get(stat, "") for stat in all_stat_names],
      assets[1]: [asset2_stats.get(stat, "") for stat in all_stat_names],
  }
  df_stats = pd.DataFrame(data, index=all_stat_names)
  df_stats.index.name = 'Stats'  # Set index name for clarity
  df_stats_reset = df_stats.reset_index()

  # Create the DataFrame as before
  df_stats = pd.DataFrame(data, index=all_stat_names)
  df_stats.index.name = 'Stats'
  df_stats_reset = df_stats.reset_index()
  #-------------------------------------------------------------------------------
  # Nice display with tabulate
  try:
      from tabulate import tabulate
      print(tabulate(df_stats_reset, headers='keys', tablefmt='fancy_grid', showindex=False, floatfmt=".6f"))
  except ImportError:
      print("Install tabulate with: pip install tabulate")
      print(df_stats_reset.to_string(index=False))
  #-------------------------------------------------------------------------------
  # Plot for EURUSD
  for asset in assets:
    #portfolio.plot(column=asset).show()
    portfolio.drawdowns.plot(column=asset).show()    # Drawdowns
    portfolio.plot_underwater(column=asset).show()   # Underwater
    portfolio.positions.plot(column=asset).show()    # Positions
  #-------------------------------------------------------------------------------
  return df_stats_reset

## Backtest

In [89]:
#-------------------------------------------------------------------------------
assets = ["EURUSD","Straddle_1M_ATMF_USD"]
close = Data[assets]
SL_Pct = 0.005
TP_Pct = 0.02
SIZE = 10
size_type='amount'
init_cash=100
fees=0.001
slippage=0.0001

#-------------------------------------------------------------------------------

orders_df1, sizes1, prices1 = Get_Orders_Futures(Data.index, Data[assets[0]], Data["Signals"], SL_Pct, TP_Pct, SIZE)
orders_df2, sizes2, prices2 = Get_Orders_Straddle_ATMF(Data.index, Data[assets[1]], Data["Signals"], SL_Pct, TP_Pct, SIZE)

print(orders_df1.tail())
print(orders_df2.tail())

sizes = pd.DataFrame({
    assets[0]: sizes1.astype(float),
    assets[1]: sizes2.astype(float)
})
prices = pd.DataFrame({
    assets[0]: prices1.astype(float),
    assets[1]: prices2.astype(float)
})
close = close.astype(float)


#-------------------------------------------------------------------------------

# Build the portfolio
portfolio = vbt.Portfolio.from_orders(
    close=close,
    size=sizes,
    price=prices,
    size_type='amount',
    freq='1D',
    init_cash=init_cash,
    fees=fees,
    slippage=slippage
)

backtesting_stats = show_backtesting_result(portfolio)
#print(backtesting_stats)

NameError: name 'Get_Orders_Straddle_ATMF' is not defined

In [85]:
SL_Pct_range = np.linspace(0.003, 0.01, 5)
TP_Pct_range = np.linspace(0.01, 0.03, 5)

# Collect all results here
all_portfolios = []

for sl in SL_Pct_range:
    for tp in TP_Pct_range:
        orders_df1, sizes1, prices1 = get_orders(Data.index, Data[assets[0]], Data["Signals"], SL_Pct, TP_Pct, SIZE)
        orders_df2, sizes2, prices2 = get_orders(Data.index, Data[assets[1]], Data["Signals"], SL_Pct, TP_Pct, SIZE)
        sizes = pd.DataFrame({assets[0]: sizes1.astype(float), assets[1]: sizes2.astype(float)})
        prices = pd.DataFrame({assets[0]: prices1.astype(float), assets[1]: prices2.astype(float)})

        portfolio = vbt.Portfolio.from_orders(
            close=close,
            size=sizes,
            price=prices,
            size_type='amount',
            freq='1D',
            init_cash=init_cash,
            fees=fees,
            slippage=slippage
        )
        # Store with params
        all_portfolios.append({
            'SL_Pct': sl,
            'TP_Pct': tp,
            'total_return': float(portfolio.total_return().mean()),  # average over assets
            'sharpe': float(portfolio.sharpe_ratio(freq='1D').mean())
        })

# Create a DataFrame of results
df_results = pd.DataFrame(all_portfolios)

# Find the best parameters
best_row = df_results.loc[df_results['total_return'].idxmax()]
print("Best parameters by total return:")
print(best_row)


Best parameters by total return:
SL_Pct          0.003000
TP_Pct          0.010000
total_return    0.016286
sharpe          0.836099
Name: 0, dtype: float64
